In [ ]:
import os
# The jupyter notebook is launched from your $HOME directory.
# Change the working directory to your username directory under /scratch/cd82
os.chdir(os.path.expandvars("/scratch/cd82/$USER/"))

# Image Classification with Convolutional Neural Networks

## Episode 02 Introduction to Image Data


In [1]:
# load the required packages
from tensorflow import keras # data and neural network
from sklearn.model_selection import train_test_split # data splitting
from tensorflow.keras.utils import img_to_array # image processing
from tensorflow.keras.utils import load_img # image processing

2025-08-19 12:09:58.154274: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-19 12:09:58.156643: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-19 12:09:58.200484: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-19 12:09:58.201471: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-08-19 12:09:59.262335: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Co

## Custom image data

### Working with Pixels

In [2]:
# specify the image path
new_img_path = "/scratch/cd82/data_icwcnn/Jabiru_TGS.JPG"

# read in the image with default arguments
new_img_pil = load_img(new_img_path)

# check the image class and size
print('Image class :', new_img_pil.__class__)
print('Image size', new_img_pil.size)

Image class : <class 'PIL.JpegImagePlugin.JpegImageFile'>
Image size (552, 573)


### Image Dimensions - Resizing

In [3]:
# read in the image and specify the target size
new_img_pil_small = load_img(new_img_path, target_size=(32,32))

# confirm the image class and size
print('Resized image class :', new_img_pil_small.__class__)
print('Resized image size', new_img_pil_small.size) 

Resized image class : <class 'PIL.Image.Image'>
Resized image size (32, 32)


## Normalisation

In [4]:
# first convert the image into an array for normalisation
new_img_arr = img_to_array(new_img_pil_small)

# confirm the image class and size
print('Converted image class  :', new_img_arr.__class__)
print('Converted image shape', new_img_arr.shape)

Converted image class  : <class 'numpy.ndarray'>
Converted image shape (32, 32, 3)


In [5]:
# inspect pixel values before and after normalisation

# extract the min, max, and mean pixel values BEFORE
print('BEFORE normalization')
print('Min pixel value ', new_img_arr.min()) 
print('Max pixel value ', new_img_arr.max())
print('Mean pixel value ', round(new_img_arr.mean(), 2))
print() # add empty line to separate output for readability

# normalize the RGB values to be between 0 and 1
new_img_arr_norm = new_img_arr / 255.0

# extract the min, max, and mean pixel values AFTER
print('AFTER normalization') 
print('Min pixel value ', new_img_arr_norm.min()) 
print('Max pixel value ', new_img_arr_norm.max())
print('Mean pixel value ', round(new_img_arr_norm.mean(), 2))

BEFORE normalization
Min pixel value  0.0
Max pixel value  255.0
Mean pixel value  87.11

AFTER normalization
Min pixel value  0.0
Max pixel value  1.0
Mean pixel value  0.34


#### Pre-existing image data

In [6]:
# load the data
(train_images, train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()

# create a list of classnames associated with each CIFAR-10 label
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

## CHALLENGE Create a function to prepare the dataset

Hint 1: Your function should accept the training images and labels as arguments

Hint 2: Use 20% split for validation and a random state of ‘42’

In [7]:
# # create a function to prepare the training dataset
# def prepare_dataset(train_images, train_labels):
    
#     # normalize the RGB values to be between 0 and 1
#     train_images= train_images /255
    
#     # one hot encode the training labels
#     train_labels= keras.utils_categorically(train_labels, num_class=length(class_names), dtype="float32")
    
#     # split the training data into training and validation set
#     train_images, val_images, train_labels, val_labels = train_test_split(
#     train_images, train_labels, test_size = 0.2, random_state=42)

#     return train_images, val_images, train_labels, val_labels

In [ ]:
## SOLUTION


## SOLUTION

In [10]:
# define a function to prepare the dataset
def prepare_dataset(train_images, train_labels):
    
    # normalize the RGB values to be between 0 and 1
    train_images = train_images / 255.0
    
    # one hot encode the training labels
    train_labels = keras.utils.to_categorical(train_labels, len(class_names))
    
    # split the training data into training and validation set
    train_images, val_images, train_labels, val_labels = train_test_split(
    train_images, train_labels, test_size = 0.2, random_state=42)

    return train_images, val_images, train_labels, val_labels

## One-hot encoding

In [11]:
# Investigate labels BEFORE one-hot encoding
print()
print('train_labels BEFORE one hot encoding')
print(train_labels)


train_labels BEFORE one hot encoding
[[6]
 [9]
 [9]
 ...
 [9]
 [1]
 [1]]


In [12]:
# prepare the dataset for training
train_images, val_images, train_labels, val_labels = prepare_dataset(train_images, train_labels)

In [13]:
# Investigate labels AFTER one-hot encoding
print()
print('train_labels AFTER one hot encoding')
print(train_labels)


train_labels AFTER one hot encoding
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 1. 0.]]


## CHALLENGE TRAINING AND VALIDATION

Inspect the training and validation sets we created.

How many samples does each set have and are the classes well balanced?

Hint: Use np.sum() on the ’*_labels’ to find out if the classes are well balanced.

In [14]:
print('Number of training set images:', train_images.shape[0])
print('Number of images in each class:\n', train_labels.sum(axis=0))

print() # add empty line to separate output for readability
print('Number of validation set images:', val_images.shape[0])
print('Nmber of images in each class:\n', val_labels.sum(axis=0))

Number of training set images: 40000
Number of images in each class:
 [4027. 4021. 3970. 3977. 4067. 3985. 4004. 4006. 3983. 3960.]

Number of validation set images: 10000
Nmber of images in each class:
 [ 973.  979. 1030. 1023.  933. 1015.  996.  994. 1017. 1040.]
